<a href="https://colab.research.google.com/github/Vivek-afk81/LLM_from_scratch/blob/main/data_preprocessing_pipeline_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing Pipeline for LLMs

**4 Main Steps**


*   Tokenization
*   Token Embeddings


*   Positional Embeddings
*   Input Embeddings





In [1]:
from google.colab import drive
drive .mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
os.chdir("/content/drive/My Drive/build_llm_from_scratch")
print(os.getcwd())

/content/drive/My Drive/build_llm_from_scratch


##Step 1: Tokenization

####Creating tokens

In [3]:
with open("The Call of the Wild.txt",'r',encoding="utf-8") as f:
  raw_text=f.read()
print("The total number of character: ",len(raw_text))
print(raw_text[:200])

The total number of character:  175584
The Call of the Wild

by Jack London




Contents

 Chapter I. Into the Primitive
 Chapter II. The Law of Club and Fang
 Chapter III. The Dominant Primordial Beast
 Chapter IV. Who Has Won to Mastersh


the regex pattern plays a very important role in splitting

In [ ]:
import re
preprocessed = re.split(r'([,.:;?_!()"\'\s]|--)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(type(preprocessed))
print(preprocessed[:30])

<class 'list'>
['The', 'Call', 'of', 'the', 'Wild', 'by', 'Jack', 'London', 'Contents', 'Chapter', 'I', '.', 'Into', 'the', 'Primitive', 'Chapter', 'II', '.', 'The', 'Law', 'of', 'Club', 'and', 'Fang', 'Chapter', 'III', '.', 'The', 'Dominant', 'Primordial']


In [ ]:
print(len(preprocessed))

36384


#### better way of tokenizing


In [ ]:
import nltk
nltk.download('punkt_tab')
preprocessed=nltk.word_tokenize(raw_text)
print(preprocessed[:300],len(preprocessed))

['The', 'Call', 'of', 'the', 'Wild', 'by', 'Jack', 'London', 'Contents', 'Chapter', 'I', '.', 'Into', 'the', 'Primitive', 'Chapter', 'II', '.', 'The', 'Law', 'of', 'Club', 'and', 'Fang', 'Chapter', 'III', '.', 'The', 'Dominant', 'Primordial', 'Beast', 'Chapter', 'IV', '.', 'Who', 'Has', 'Won', 'to', 'Mastership', 'Chapter', 'V.', 'The', 'Toil', 'of', 'Trace', 'and', 'Trail', 'Chapter', 'VI', '.', 'For', 'the', 'Love', 'of', 'a', 'Man', 'Chapter', 'VII', '.', 'The', 'Sounding', 'of', 'the', 'Call', 'Chapter', 'I', '.', 'Into', 'the', 'Primitive', '“', 'Old', 'longings', 'nomadic', 'leap', ',', 'Chafing', 'at', 'custom', '’', 's', 'chain', ';', 'Again', 'from', 'its', 'brumal', 'sleep', 'Wakens', 'the', 'ferine', 'strain.', '”', 'Buck', 'did', 'not', 'read', 'the', 'newspapers', ',', 'or', 'he', 'would', 'have', 'known', 'that', 'trouble', 'was', 'brewing', ',', 'not', 'alone', 'for', 'himself', ',', 'but', 'for', 'every', 'tide-water', 'dog', ',', 'strong', 'of', 'muscle', 'and', 'with'

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


##Step 2  creating token ids

complete training dataset ---> tokenized text---> Vocabulary

1. Tokenization breaks
down the input text
into individual tokens.

2. Each unique token is
added to the vocabulary
in alphabetical order.

In [ ]:
all_words= sorted(set(preprocessed))
vocab_size=len(all_words)
vocab_size

5131

In [ ]:
#Assiging an integer to each and every token
vocab={token:integer for integer,token in enumerate(all_words)}


In [ ]:
for i,item in enumerate(vocab.items()):
  print(item)
  if i>=50:
    break

('!', 0)
('(', 1)
(')', 2)
(',', 3)
('.', 4)
('...', 5)
('1897', 6)
(':', 7)
(';', 8)
('?', 9)
('A', 10)
('A-a-ah', 11)
('About', 12)
('Across', 13)
('After', 14)
('Again', 15)
('Ah', 16)
('Air-holes', 17)
('Alaska', 18)
('Alaska.', 19)
('Alaskan', 20)
('Alice', 21)
('All', 22)
('Alpine', 23)
('Also', 24)
('Always', 25)
('Among', 26)
('An', 27)
('And', 28)
('Angry', 29)
('Another', 30)
('Answers', 31)
('Apt', 32)
('Arctic', 33)
('As', 34)
('Ask', 35)
('Association', 36)
('At', 37)
('Back', 38)
('Bar', 39)
('Barge', 40)
('Barracks', 41)
('Barrens', 42)
('Bay', 43)
('Be', 44)
('Beast', 45)
('Because', 46)
('Before', 47)
('Being', 48)
('Bellying', 49)
('Bench', 50)


##Train a Hugging Face BPE Tokenizer

We will use **BPE** because Instead of treating each word as a token, BPE starts from characters and repeatedly merges the most frequent character pairs. This allows the tokenizer to represent common words as single tokens while still being able to compose rare or unseen words from subwords.

example

"lower", "lowest"

→ l o w e r
→ l o w e s t

Frequent merges:
* l+o → lo
* lo+w → low
* e+r → er
* e+s → es


In [ ]:
#Rust-backed, production-grade library
!pip install -q tokenizers

###Create a BPE tokenizer

In [ ]:
#Initializing the tokenizer

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

In [ ]:
tokenizer=Tokenizer(BPE(unk_token="[UNK]"))  # a fallback for the unseen tokens


####Pre-tokenization

Pre-tokenization defines initial splits


Here: split on whitespace before BPE merges

BPE does NOT work on raw text directly — it works on pre-tokenized units

In [ ]:
tokenizer.pre_tokenizer=Whitespace()
tokenizer

Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[], normalizer=None, pre_tokenizer=Whitespace(), post_processor=None, decoder=None, model=BPE(dropout=None, unk_token="[UNK]", continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={}, merges=[]))

In [ ]:
##Defining how the vocubalary will be learned
"""
Word-level vocab (5131): Every distinct word is counted.

BPE vocab (5000): We're asking the tokenizer to learn a smaller, more
efficient set of subword tokens."""
trainer=BpeTrainer(
    vocab_size=5000,
    special_tokens=["[UNK]","[PAD]","[BOS]","[EOS]"]
)
trainer

BpeTrainer(BpeTrainer(min_frequency=0, vocab_size=5000, show_progress=True, special_tokens=[AddedToken(content="[UNK]", single_word=False, lstrip=False, rstrip=False, normalized=False, special=True), AddedToken(content="[PAD]", single_word=False, lstrip=False, rstrip=False, normalized=False, special=True), AddedToken(content="[BOS]", single_word=False, lstrip=False, rstrip=False, normalized=False, special=True), AddedToken(content="[EOS]", single_word=False, lstrip=False, rstrip=False, normalized=False, special=True)], limit_alphabet=None, initial_alphabet=[], continuing_subword_prefix=None, end_of_word_suffix=None, max_token_length=None, words={}))

In [ ]:
##Training the tokenizer

tokenizer.train(
    files=["The Call of the Wild.txt"],
    trainer=trainer
)

In [ ]:
tokenizer.get_vocab_size()

5000

In [ ]:
tokenizer.get_vocab()

{'matted': 2738,
 'forty': 1317,
 'incl': 2106,
 'Government': 3002,
 'oh': 3195,
 'ranging': 2757,
 'Tho': 3073,
 'romp': 4976,
 'quiet': 1949,
 'sweater': 1017,
 'Iri': 4317,
 'i': 50,
 'ropes': 4986,
 'worse': 1647,
 'Jo': 346,
 'llab': 4965,
 'listening': 1984,
 'todon': 4889,
 '[UNK]': 0,
 'capsized': 3957,
 'hind': 452,
 'ge': 153,
 'proceed': 3636,
 'atten': 4876,
 'fixed': 2687,
 'Brien': 3000,
 'grow': 552,
 'tink': 4691,
 'eye': 1910,
 'winter': 1207,
 'instinctively': 4232,
 'VI': 3078,
 'almost': 1454,
 'ades': 4959,
 'ure': 371,
 'heal': 4752,
 'mber': 2483,
 'atif': 4878,
 'droop': 2768,
 'Mat': 1091,
 'first': 457,
 'train': 1329,
 'others': 1065,
 'acher': 2199,
 'crying': 2313,
 'desperate': 4019,
 'ram': 2582,
 'ally': 659,
 'cap': 1463,
 'Die': 4286,
 'l': 53,
 'Yep': 4386,
 'able': 398,
 'trotted': 3237,
 'nu': 4587,
 'We': 2035,
 'Me': 4333,
 'pal': 3202,
 'brown': 3603,
 'eagerness': 2319,
 'coils': 3480,
 'thump': 4835,
 'silence': 1958,
 'ided': 894,
 'survi': 2

####Encoding

In [ ]:
encoded=tokenizer.encode("the dog ran quickly.")
for token_id in encoded.ids:
  print(f"{token_id}: {tokenizer.decode([token_id])}")



77: the
163: dog
185: ran
1004: quick
107: ly
9: .


####Saving the tokenizer


In [ ]:
# tokenizer.save("bpe_tokenizer.json") i have saved this once,this will provide us with reproducibe results

###INPUT -TARGET CREATION

We will now implementing a GPT-style training data pipeline by converting a continuous token stream into overlapping fixed-length context windows and generating input–target pairs for next-token prediction.

In [ ]:
from tokenizers import Tokenizer

tokenizer=Tokenizer.from_file("bpe_tokenizer.json")

encoded=tokenizer.encode(raw_text)
token_ids=encoded.ids

print("total number of tokens: ",len(token_ids))

total number of tokens:  40213


In [ ]:
##Chunk the token streams

def create_chunks(token_ids,context_length,stride):
  chunks=[]
  i=0

  while i + context_length+1<=len(token_ids):
    chunk=token_ids[i: i+context_length+1]
    chunks.append(chunk)
    i+=stride

  return chunks

####Create Input Target Pairs

In [ ]:
def create_input_target_pairs(chunks):
  inputs=[]
  targets=[]

  for chunk in chunks:
    inputs.append(chunk[:-1])
    targets.append(chunk[1:])

  return inputs,targets

In [ ]:
context_length=128 #toy model
stride =64 # 50% overlap

chunks=create_chunks(token_ids,context_length,stride)
inputs,targets=create_input_target_pairs(chunks)

print("Number of samples:", len(inputs))
print("Input shape:", len(inputs[0]))
print("Target shape:", len(targets[0]))

Number of samples: 627
Input shape: 128
Target shape: 128


In [ ]:
#testing if this works decoding one sample

sample_input=tokenizer.decode(inputs[0])
sample_target=tokenizer.decode(targets[0])

print("INPUT:")
print(sample_input)

print("\nTARGET:")
print(sample_target)  #works like a charm

INPUT:
The Call of the Wild by Jack Lond on Con tents Chapter I . Into the Primitive Chapter II . The Law of Club and Fang Chapter III . The Dominant Primordial Beast Chapter IV . Who Has Won to Mastership Chapter V . The Toil of Trace and Trail Chapter VI . For the Love of a Man Chapter VII . The Sounding of the Call Chapter I . Into the Primitive “ Old long ings nomad ic leap , Ch af ing at custom ’ s chain ; Again from its bru mal sleep Waken s the fer ine strain .” Buck did not read the newspapers , or he would have known that trouble was bre wing , not alone for himself , but for

TARGET:
Call of the Wild by Jack Lond on Con tents Chapter I . Into the Primitive Chapter II . The Law of Club and Fang Chapter III . The Dominant Primordial Beast Chapter IV . Who Has Won to Mastership Chapter V . The Toil of Trace and Trail Chapter VI . For the Love of a Man Chapter VII . The Sounding of the Call Chapter I . Into the Primitive “ Old long ings nomad ic leap , Ch af ing at custom ’ s cha

### PyTorch Dataset & DataLoader

Wrap our (input, target) pairs into a clean, reusable Dataset, then load them efficiently with a DataLoader.

In [ ]:
import torch
from torch.utils.data import Dataset


In [ ]:
class llm_text_dataset(Dataset):
  def __init__(self,inputs,targets):
    assert len(inputs)==len(targets),"Inputs and targets must have same length othervise ,its time for a whole lot of checking previous cells"
    self.inputs=inputs
    self.targets=targets

  def __len__(self):
    return len(self.inputs)

  def __getitem__(self,idx):
    x=torch.tensor(self.inputs[idx],dtype=torch.long)
    y=torch.tensor(self.targets[idx],dtype=torch.long)
    return x,y

In [ ]:
dataset=llm_text_dataset(inputs,targets)
print("Total training samples:", len(dataset))

Total training samples: 627


In [ ]:
from torch.utils.data import DataLoader


In [ ]:
batch_size=32

dataloader=DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,   # prevents memorization
    drop_last=True
)

In [ ]:
##Lets check one batch and see how it performs

batch_1,batch_2=next(iter(dataloader))

print("batch input shape: ",batch_1.shape)
print("batch target shape: ",batch_2.shape)

batch input shape:  torch.Size([32, 128])
batch target shape:  torch.Size([32, 128])


#####Decode one sample

In [ ]:
sample_input_text = tokenizer.decode(batch_1[0].tolist())
sample_target_text = tokenizer.decode(batch_2[0].tolist())

print("INPUT:")
print(sample_input_text)

print("\nTARGET:")
print(sample_target_text)


INPUT:
, as the Yeehats were hunting it , on the flanks of the mig rating moose , the wolf pack had at last crossed over from the land of streams and timber and invaded Buck ’ s valley . Into the clearing where the moonlight streamed , they pou red in a silvery flood ; and in the centre of the clearing stood Buck , motionless as a statue , waiting their coming . They were aw ed , so still and large he stood , and a moment ’ s pause fell , till the bold est one leaped straight for him . Like a flash Buck struck , breaking the neck . Then he stood , without movement , as before , the stricken wolf rolling

TARGET:
as the Yeehats were hunting it , on the flanks of the mig rating moose , the wolf pack had at last crossed over from the land of streams and timber and invaded Buck ’ s valley . Into the clearing where the moonlight streamed , they pou red in a silvery flood ; and in the centre of the clearing stood Buck , motionless as a statue , waiting their coming . They were aw ed , so sti